In [0]:
dbutils.widgets.text("databaseName", "fraud_gold")
dbutils.widgets.text("storageName", "stfrauddatabricks")

storage_account_name = dbutils.widgets.get("storageName")
schema_name = dbutils.widgets.get("databaseName")

container_gold = "gold"
project_folder = "fraud_detection"

gold_base_path = f"abfss://{container_gold}@{storage_account_name}.dfs.core.windows.net/{project_folder}"


In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

spark.sql("""
SELECT 
  current_catalog() AS catalogo_actual,
  current_schema() AS schema_actual
""").show(truncate=False)

In [0]:
# definir catálogo y schema

catalog_name = "dbw_fraud_detection"
schema_name = "fraud_gold"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
spark.sql(f"USE SCHEMA {schema_name}")

print("GOLD path:", gold_base_path)
print(f"Catálogo usado: {catalog_name}")
print(f"Base de datos / schema activo: {schema_name}")

In [0]:


gold_tables = {
    "gold_kpi_fraud_summary": f"{gold_base_path}/gold_kpi_fraud_summary_delta",
    "gold_fraud_by_transaction_type": f"{gold_base_path}/gold_fraud_by_transaction_type_delta",
    "gold_fraud_by_risk_level": f"{gold_base_path}/gold_fraud_by_risk_level_delta",
    "gold_fraud_by_amount_category": f"{gold_base_path}/gold_fraud_by_amount_category_delta",
    "gold_fraud_by_profile_risk": f"{gold_base_path}/gold_fraud_by_profile_risk_delta",
    "gold_top_suspicious_accounts": f"{gold_base_path}/gold_top_suspicious_accounts_delta",
    "gold_fraud_by_step": f"{gold_base_path}/gold_fraud_by_step_delta",
    "gold_dashboard_executive_summary": f"{gold_base_path}/gold_dashboard_executive_summary_delta"
}

for table_name, table_path in gold_tables.items():
    spark.sql(f"DROP TABLE IF EXISTS {schema_name}.{table_name}")

    spark.sql(f"""
        CREATE TABLE {schema_name}.{table_name}
        USING DELTA
        LOCATION '{table_path}'
    """)

    print(f"✅ Tabla registrada: {schema_name}.{table_name}")

In [0]:
# COMMAND ----------

display(spark.sql(f"""
SELECT *
FROM {catalog_name}.{schema_name}.gold_kpi_fraud_summary
"""))

In [0]:
display(spark.sql(f"""
SELECT *
FROM {schema_name}.gold_kpi_fraud_summary
"""))

In [0]:
display(spark.sql(f"""
SELECT 
    transaction_type,
    risk_level,
    total_transactions,
    fraud_transactions,
    fraud_amount,
    fraud_rate_pct
FROM {schema_name}.gold_fraud_by_transaction_type
ORDER BY fraud_transactions DESC
"""))

In [0]:
display(spark.sql(f"""
SELECT 
    origin_account_id,
    origin_customer_segment,
    origin_kyc_status,
    total_transactions,
    fraud_transactions,
    fraud_amount,
    suspicious_account_score
FROM {schema_name}.gold_top_suspicious_accounts
ORDER BY suspicious_account_score DESC
LIMIT 20
"""))